# Fine-tune LLaMA 3.2 3B for Ido↔Esperanto (QLoRA / PEFT+TRL)

One bidirectional model on the dataset built by `llm/data/make_dataset.py`,
pulled from GitHub. Standard `transformers + peft + trl` QLoRA — **no Unsloth**
(its latest release is incompatible with Kaggle's transformers 5.x). Runs on
**Kaggle** (disconnect-proof) or Colab.

## Kaggle — recommended (survives closing the browser)
1. kaggle.com → **Create → New Notebook → File → Import Notebook** → paste this
   notebook's URL (`https://github.com/komapc/ido-epo-llm/blob/main/finetune_colab.ipynb`)
   or upload the `.ipynb`.
2. Right sidebar **Settings**: Accelerator = **GPU T4 x2**; **Internet = On**
   (needs a phone-verified account — required for pip + git clone).
3. **Save Version → Save & Run All (Commit)**. It runs to completion on Kaggle's
   servers — you can close the tab. When it finishes, open that version →
   **Output** tab → download `preds.jsonl` and `ido-epo-lora.zip` (in `/kaggle/working/`).

Data repo is **public**, so no token/secret is needed.

## Colab — alternative
Runtime → Change runtime type → T4 GPU → Run all.

Baseline to beat (Apertium on the real test split): **chrF 57.0 / BLEU 18.1**,
especially the 60% of inputs where Apertium emits a `*`/`#`/`@` failure marker.

In [ ]:
%%capture
# Standard QLoRA stack — NO Unsloth. We dropped Unsloth because its latest
# release requires transformers>=5, but its patched training step crashes on
# transformers 5.x ("'int' object has no attribute 'mean'"), and pip can't
# resolve an older compatible Unsloth. Plain transformers+peft+trl works fine on
# Kaggle's stock transformers 5.x. Just make sure bitsandbytes/peft/trl are current.
!pip install -q -U bitsandbytes peft trl accelerate datasets

In [ ]:
# --- Pull dataset from the repo ---------------------------------------------
import os, subprocess

# Kaggle gives "T4 x2"; HF Trainer auto-wraps in DataParallel across both GPUs,
# which breaks bitsandbytes 4-bit (CUBLAS_STATUS_EXECUTION_FAILED). Use ONE GPU.
# Must be set before torch initializes CUDA (i.e. before the model cell).
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

REPO_URL    = "https://github.com/komapc/ido-epo-llm.git"  # change to your repo
DATA_SUBDIR = "data"          # path inside the repo holding train/val/test.jsonl
BRANCH      = "main"
# For a PRIVATE repo, set a fine-grained PAT (Colab: 🔑 Secrets -> GH_TOKEN):
GH_TOKEN    = os.environ.get("GH_TOKEN", "")
try:
    from google.colab import userdata
    GH_TOKEN = GH_TOKEN or userdata.get("GH_TOKEN")
except Exception:
    pass

clone_url = REPO_URL
if GH_TOKEN:
    clone_url = REPO_URL.replace("https://", f"https://{GH_TOKEN}@")

REPO_DIR = REPO_URL.rstrip('/').split('/')[-1][:-4]  # 'ido-epo-llm'
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, clone_url], check=True)

DATA_DIR = os.path.join(REPO_DIR, DATA_SUBDIR)
for split in ("train", "val", "test"):
    p = os.path.join(DATA_DIR, f"{split}.jsonl")
    assert os.path.exists(p), f"missing {p} — check REPO_URL/DATA_SUBDIR"
    print(split, sum(1 for _ in open(p)), "rows")

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # single GPU (see clone cell) — must precede torch
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MAX_SEQ = 512
BASE = "unsloth/Llama-3.2-3B-Instruct"  # ungated mirror of meta-llama/Llama-3.2-3B-Instruct

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# torch_dtype=float16 is REQUIRED on T4: it has no bf16, and we train with
# fp16=True. If the model loads bf16 weights (Llama's config default), the fp16
# grad scaler crashes ("...unscale_ not implemented for 'BFloat16'").
model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, device_map={"": 0}, torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
))

# CRITICAL on T4: torch_dtype on the BASE is not enough. peft creates the LoRA
# adapter weights in Llama's *config* dtype (bfloat16), regardless of how we
# loaded the base. Those are the only trainable params, so their grads are bf16
# and the fp16 scaler crashes at clip_grad_norm. Force every bf16 param to fp16
# so no bf16 gradient can exist; fp32 params (layer norms) are left untouched.
n_cast = 0
for _p in model.parameters():
    if _p.dtype == torch.bfloat16:
        _p.data = _p.data.to(torch.float16)
        n_cast += 1
print(f"cast {n_cast} bf16 params -> fp16")
dtypes = {str(p.dtype) for p in model.parameters() if p.requires_grad}
print("trainable param dtypes:", dtypes)  # expect only float32 / float16, NO bfloat16

model.print_trainable_parameters()

In [ ]:
import json
from datasets import Dataset

def load_jsonl(name):
    path = os.path.join(DATA_DIR, name)
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

train_rows = load_jsonl("train.jsonl")

# Oversample by integer weight so real pairs (w=3) are seen 3x vs synthetic (w=1).
expanded = []
for r in train_rows:
    expanded.extend([r] * int(r.get("weight", 1)))
print(f"{len(train_rows)} unique -> {len(expanded)} after weighting")

def to_text(r):
    msgs = [
        {"role": "system", "content": "You are a precise Ido↔Esperanto translator. Output only the translation."},
        {"role": "user", "content": f"{r['instruction']}\n{r['input']}"},
        {"role": "assistant", "content": r["output"]},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False)

ds = Dataset.from_list([{"text": to_text(r)} for r in expanded]).shuffle(seed=42)
print(ds[0]["text"][:400])

In [ ]:
from trl import SFTTrainer, SFTConfig

# Standard TRL QLoRA, 1 epoch (~3h on a T4 without Unsloth's speedups).
# If you hit CUDA OOM, drop per_device_train_batch_size to 2 and raise
# gradient_accumulation_steps to 16 (keeps the effective batch at 32).
args = SFTConfig(
    output_dir="outputs",
    per_device_train_batch_size=4, gradient_accumulation_steps=8,
    warmup_ratio=0.03, num_train_epochs=1, learning_rate=2e-4,
    fp16=True, bf16=False, optim="paged_adamw_8bit",
    logging_steps=25, weight_decay=0.01, lr_scheduler_type="cosine", seed=42,
    save_strategy="epoch", save_only_model=True, report_to="none",
    dataset_text_field="text", packing=False,
)
# processing_class (new TRL) vs tokenizer (old TRL) — support both.
try:
    trainer = SFTTrainer(model=model, train_dataset=ds, args=args, processing_class=tokenizer)
except TypeError:
    trainer = SFTTrainer(model=model, train_dataset=ds, args=args, tokenizer=tokenizer)
trainer.train()

# Save the adapter IMMEDIATELY — protects the 3h run if prediction below hiccups.
model.save_pretrained("ido-epo-lora")
tokenizer.save_pretrained("ido-epo-lora")
print("✅ adapter saved to ido-epo-lora/")

In [ ]:
# Generate predictions on the held-out real test split -> preds.jsonl
import json
model.config.use_cache = True          # re-enable KV cache (off during training) for fast gen
try:
    model.gradient_checkpointing_disable()
except Exception:
    pass
model.eval()
test_rows = load_jsonl("test.jsonl")

def translate(instruction, text):
    msgs = [
        {"role": "system", "content": "You are a precise Ido↔Esperanto translator. Output only the translation."},
        {"role": "user", "content": f"{instruction}\n{text}"},
    ]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(input_ids=ids, max_new_tokens=128, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

with open("preds.jsonl", "w", encoding="utf-8") as f:
    for i, r in enumerate(test_rows):
        try:
            pred = translate(r["instruction"], r["input"])
        except Exception as e:
            pred = ""
            if i < 3:
                print("gen error:", e)
        f.write(json.dumps({"output": pred}, ensure_ascii=False) + "\n")
        f.flush()
        if i < 8:
            print(f"{r['input']}\n  ref : {r['output']}\n  pred: {pred}\n")
print("wrote preds.jsonl — download it and run: python3 eval/eval_chrf.py --apertium --pred preds.jsonl")

In [ ]:
# Save the LoRA adapter (small) -> download from /kaggle/working
model.save_pretrained("ido-epo-lora")
tokenizer.save_pretrained("ido-epo-lora")
!zip -r ido-epo-lora.zip ido-epo-lora >/dev/null && echo 'adapter saved to ido-epo-lora.zip'

# To serve later: load base + this adapter with peft, or merge:
#   from peft import PeftModel
#   merged = PeftModel.from_pretrained(base_model, "ido-epo-lora").merge_and_unload()